In [2]:
!pip install lightkurve -q

import time
import numpy as np
import pandas as pd
import lightkurve as lk

def retry(action, attempts=5, wait_seconds=60):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as error:
            print(f"Attempt {attempt} of {attempts} failed: {type(error).__name__}")
            if attempt == attempts:
                raise
            time.sleep(wait_seconds)

search_result = retry(lambda: lk.search_lightcurve("WASP-18", mission="TESS", author="SPOC", exptime=120))
lc_sap = retry(lambda: search_result[0].download(flux_column="sap_flux"))
lc_pdcsap = retry(lambda: search_result[0].download(flux_column="pdcsap_flux"))

table = pd.DataFrame({
    "time_btjd": lc_sap.time.value,
    "sap_flux": lc_sap.flux.value,
    "sap_flux_err": lc_sap.flux_err.value,
    "pdcsap_flux": lc_pdcsap.flux.value,
    "pdcsap_flux_err": lc_pdcsap.flux_err.value,
})

from google.colab import drive
drive.mount("/content/drive")
table.to_csv("/content/drive/MyDrive/wasp18_sector2.csv", index=False)

print("Rows saved:", len(table))
print("Original file name:", lc_sap.meta.get("FILENAME"))

Mounted at /content/drive
Rows saved: 18299
Original file name: /root/.lightkurve/cache/mastDownload/TESS/tess2018234235059-s0002-0000000100100827-0121-s/tess2018234235059-s0002-0000000100100827-0121-s_lc.fits
